PROBLEM STATEMENT 5
Prepare a dashboard to provide an interface for user to  perform the following operaations:
1. Named-Entity Relationship
2. POS Tagging
3. POS Distribution
4. Lemmatization
5. Stemming
6. Morphology
7. Dependencies - style=dep

In [4]:
!pip install streamlit spacy nltk plotly
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
!pip install dash jupyter-dash spacy nltk plotly -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
!pip install dash-bootstrap-components -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 5.1 MB/s eta 0:00:00


In [18]:
import spacy
import nltk
from nltk.stem import PorterStemmer
from collections import Counter
import pandas as pd
import plotly.express as px
from spacy import displacy

from dash import Dash
from dash import dcc, html, Input, Output, State, dash_table
import dash_bootstrap_components as dbc
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

nltk.download('punkt', quiet=True)
nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()

EXAMPLE_TEXT = ("On Monday, September 15, 2025, at 10:00 AM EST, Dr. Helena Vance "
                 "announced that the company's new AI research lab in San Francisco "
                 "would open next quarter, aiming to hire over 200 engineers.")

app = JupyterDash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    html.H2("NLP Explorer", className="mt-3"),
    html.P("Interactive NLP Pipeline Analyzer", className="text-muted"),

    dcc.Textarea(
        id="input-text", value=EXAMPLE_TEXT,
        style={"width": "100%", "height": 120},
        placeholder="Enter or paste a paragraph here..."
    ),
    dbc.Button("Analyze Text", id="analyze-btn", color="primary", className="mt-2 mb-3", n_clicks=0),


    html.Div(id="stat-cards", className="mb-3"),

    dcc.Tabs(id="tabs", value="overview", children=[
        dcc.Tab(label="Overview", value="overview"),
        dcc.Tab(label="Tokens", value="tokens"),
        dcc.Tab(label="POS Tagging", value="pos"),
        dcc.Tab(label="POS Distribution", value="pos-dist"),
        dcc.Tab(label="NER", value="ner"),
        dcc.Tab(label="Lemmatization", value="lemma"),
        dcc.Tab(label="Stemming", value="stem"),
        dcc.Tab(label="Morphology", value="morph"),
        dcc.Tab(label="Dependencies", value="dep"),
    ]),
    html.Div(id="tab-content", className="mt-3 mb-5"),

    dcc.Store(id="doc-store")  # cache analysis result across tab switches
], fluid=True)


def analyze_text(text):
    doc = nlp(text)
    return {
        "tokens": [t.text for t in doc],
        "pos": [(t.text, t.pos_, t.tag_) for t in doc],
        "lemma": [(t.text, t.lemma_) for t in doc],
        "morph": [(t.text, t.pos_, str(t.morph)) for t in doc],
        "ents": [(e.text, e.label_) for e in doc.ents],
        "ner_html": displacy.render(doc, style="ent", page=False),
        "dep_svg": displacy.render(doc, style="dep", options={"compact": True, "distance": 100}),
        "n_tokens": len(doc),
        "n_words": len([t for t in doc if not t.is_punct and not t.is_space]),

        "n_sents": len(list(doc.sents)),
        "n_ents": len(doc.ents),
        "n_stop": len([t for t in doc if t.is_stop]),
        "n_punct": len([t for t in doc if t.is_punct]),
    }


@app.callback(
    Output("doc-store", "data"),
    Output("stat-cards", "children"),
    Input("analyze-btn", "n_clicks"),
    State("input-text", "value"),
    prevent_initial_call=False
)
def run_analysis(n_clicks, text):
    if not text:
        return {}, html.Div("Enter text and click Analyze.")
    data = analyze_text(text)

    def card(label, value):
        return dbc.Col(dbc.Card(dbc.CardBody([
            html.Div(label, className="text-muted small"),
            html.H4(str(value))
        ])), width=2)

    cards = dbc.Row([
        card("Tokens", data["n_tokens"]),
        card("Words", data["n_words"]),
        card("Sentences", data["n_sents"]),
        card("Entities", data["n_ents"]),
        card("Stopwords", data["n_stop"]),
        card("Punctuation", data["n_punct"]),

    ])
    return data, cards


@app.callback(
    Output("tab-content", "children"),
    Input("tabs", "value"),
    Input("doc-store", "data")
)
def render_tab(tab, data):
    if not data:
        return html.Div("Click 'Analyze Text' to begin.")

    if tab == "overview":
        return html.Div([
            html.P(f"{data['n_tokens']} tokens, {data['n_words']} words, "
                   f"{data['n_sents']} sentences, {data['n_ents']} entities found.")
        ])

    elif tab == "tokens":
        df = pd.DataFrame({"#": range(1, len(data["tokens"]) + 1), "Token": data["tokens"]})
        return dash_table.DataTable(df.to_dict("records"), page_size=20,
                                     style_table={"overflowX": "auto"})

    elif tab == "pos":
        df = pd.DataFrame(data["pos"], columns=["Token", "POS (coarse)", "Tag (fine)"])
        return dash_table.DataTable(df.to_dict("records"), page_size=20,
                                     style_table={"overflowX": "auto"})

    elif tab == "pos-dist":
        counts = Counter(p[1] for p in data["pos"])
        df = pd.DataFrame(counts.items(), columns=["POS", "Count"]).sort_values("Count")

        fig = px.bar(df, x="Count", y="POS", orientation="h", title="POS Distribution")
        fig.update_layout(height=500)
        return dcc.Graph(figure=fig)

    elif tab == "ner":
        if not data["ents"]:
            return html.Div("No named entities found.")
        return html.Iframe(srcDoc=data["ner_html"],
                            style={"width": "100%", "height": "300px", "border": "none"})

    elif tab == "lemma":
        df = pd.DataFrame(data["lemma"], columns=["Token", "Lemma"])
        return dash_table.DataTable(df.to_dict("records"), page_size=20,
                                     style_table={"overflowX": "auto"})

    elif tab == "stem":
        df = pd.DataFrame({"Token": data["tokens"],
                            "Stem": [stemmer.stem(t) for t in data["tokens"]]})
        return dash_table.DataTable(df.to_dict("records"), page_size=20,
                                     style_table={"overflowX": "auto"})

    elif tab == "morph":
        df = pd.DataFrame(data["morph"], columns=["Token", "POS", "Morphology"])
        return dash_table.DataTable(df.to_dict("records"), page_size=20,
                                     style_table={"overflowX": "auto"})

    elif tab == "dep":
        return html.Iframe(srcDoc=data["dep_svg"],
                            style={"width": "100%", "height": "450px", "border": "none"})


app.run(jupyter_mode="inline", jupyter_height=900)

<IPython.core.display.Javascript object>